# Data Preprocessing (Task 2, step 1)
**CM2604 Machine Learning, Part 01**

The cleaning, splitting and encoding logic lives in `src/preprocessing.py` so every model uses exactly the same steps. This notebook runs it and explains each decision.

In [1]:
import sys
sys.path.append("../src")   # lets us import src/preprocessing.py

import pandas as pd
from preprocessing import load_and_clean, split_data, build_preprocessor, NUMERIC_COLS

raw = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Raw shape:", raw.shape)

Raw shape: (7043, 21)


## 1. Cleaning decisions
| Issue found in EDA | Decision | Reason |
|---|---|---|
| `customerID` | Drop | Unique ID, no predictive value, and would let models memorise rows |
| `TotalCharges` stored as text, 11 blanks | Convert to numeric, fill blanks with 0 | All 11 have `tenure = 0` (not billed yet), so 0 is the true value. Keeps all 7,043 rows |
| "No internet service" / "No phone service" | Recode to "No" | Already captured by `InternetService` / `PhoneService`, so the extra category is redundant |
| `SeniorCitizen` is 0/1 | Treat as categorical | It is a yes/no flag, not a quantity, so it should not be scaled |
| `Churn` is Yes/No | Map to 1/0 | Models need a numeric target (1 = churned) |

In [2]:
df = load_and_clean()
print("Clean shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
df.head()

Clean shape: (7043, 20)
Missing values: 0


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,No,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,No,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [3]:
# Check the recoding worked: only Yes/No remain
for c in ["OnlineSecurity", "TechSupport", "MultipleLines", "SeniorCitizen"]:
    print(c, sorted(df[c].unique()))
print("Churn values:", sorted(df["Churn"].unique()))

OnlineSecurity ['No', 'Yes']
TechSupport ['No', 'Yes']
MultipleLines ['No', 'Yes']
SeniorCitizen ['No', 'Yes']
Churn values: [np.int64(0), np.int64(1)]


## 2. Train/test split
- **80% train / 20% test**, with `random_state=42` so results are reproducible.
- **Stratified** on `Churn`, so both sets keep the same churn ratio (about 26.5%). This matters because the classes are imbalanced.
- The split happens **before** scaling and encoding, so the test set stays unseen (no data leakage).

In [4]:
X_train, X_test, y_train, y_test = split_data(df)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print(f"Churn rate  train: {y_train.mean():.4f}   test: {y_test.mean():.4f}   full: {df['Churn'].mean():.4f}")

Train: (5634, 19) | Test: (1409, 19)
Churn rate  train: 0.2654   test: 0.2654   full: 0.2654


## 3. Encoding and scaling
- **Numeric** (`tenure`, `MonthlyCharges`, `TotalCharges`): `StandardScaler`. Essential for **KNN** (distance-based) and **Logistic Regression** (regularisation and convergence). Harmless for Random Forest.
- **Categorical**: `OneHotEncoder(drop="first")`. One-hot avoids implying a false order between categories. Dropping the first level removes perfect multicollinearity, which helps Logistic Regression.
- Both are combined in one `ColumnTransformer`. It is **fitted on the training data only**, then applied to the test data.

In [5]:
preprocessor = build_preprocessor(X_train)

X_train_t = preprocessor.fit_transform(X_train)   # learn means/categories from TRAIN only
X_test_t = preprocessor.transform(X_test)          # apply them to TEST

feature_names = preprocessor.get_feature_names_out()
print("Features before:", X_train.shape[1], " -> after encoding:", X_train_t.shape[1])
print("Train:", X_train_t.shape, " Test:", X_test_t.shape)
print(list(feature_names))

Features before: 19  -> after encoding: 23
Train: (5634, 23)  Test: (1409, 23)
['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges', 'cat__gender_Male', 'cat__SeniorCitizen_Yes', 'cat__Partner_Yes', 'cat__Dependents_Yes', 'cat__PhoneService_Yes', 'cat__MultipleLines_Yes', 'cat__InternetService_Fiber optic', 'cat__InternetService_No', 'cat__OnlineSecurity_Yes', 'cat__OnlineBackup_Yes', 'cat__DeviceProtection_Yes', 'cat__TechSupport_Yes', 'cat__StreamingTV_Yes', 'cat__StreamingMovies_Yes', 'cat__Contract_One year', 'cat__Contract_Two year', 'cat__PaperlessBilling_Yes', 'cat__PaymentMethod_Credit card (automatic)', 'cat__PaymentMethod_Electronic check', 'cat__PaymentMethod_Mailed check']


In [6]:
# Sanity check: scaled numeric columns should have mean ~0 and std ~1 on TRAIN
check = pd.DataFrame(X_train_t[:, :3], columns=NUMERIC_COLS)
check.agg(["mean", "std"]).round(3)

,tenure,MonthlyCharges,TotalCharges
mean,-0.0,-0.0,0.0
std,1.0,1.0,1.0


## 4. Class imbalance
The preprocessing does **not** resample the data. Instead, imbalance is handled at modelling time (for example `class_weight="balanced"` and metrics such as recall, F1 and ROC-AUC), so the test set keeps the real-world churn rate.

## 5. Why a Pipeline later
In the modelling notebook the preprocessor will be placed inside a scikit-learn `Pipeline` with each model. During cross-validation and hyperparameter tuning, scaling and encoding are then re-fitted on each training fold only, which prevents leakage.